In [22]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, classification_report
)

# 1. Load and label
df = pd.read_csv("merged_features.csv")
df["has_adhd"] = df["folder"].str.contains("ADHD", case=False).astype(int)

# 2. Split into features/target and then train/test
X = df.drop(columns=["folder", "filename", "has_adhd"])
y = df["has_adhd"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 3. Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# 4. LDA on training, then transform both splits
lda = LinearDiscriminantAnalysis(n_components=1)
X_train_lda = lda.fit_transform(X_train_scaled, y_train)
X_test_lda  = lda.transform(X_test_scaled)

# 5. Grid‐search RBF‐SVM on LDA features
param_grid = {
    "C": [1, 5, 10],
    "kernel": ["rbf"],
    "gamma": [0.001, 0.005, 0.01],
    "class_weight": ["balanced", None]
}
svc = SVC(probability=True, random_state=42)
grid = GridSearchCV(svc, param_grid, cv=5, scoring="f1", n_jobs=-1, verbose=1)
grid.fit(X_train_lda, y_train)
best_svc = grid.best_estimator_

# 6. Evaluate on hold‐out test set
y_pred  = best_svc.predict(X_test_lda)
y_proba = best_svc.predict_proba(X_test_lda)[:, 1]

print("=== Best Hyperparameters ===")
print(grid.best_params_, "\n")

print("=== Test Set Performance ===")
print(f"Accuracy  : {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision : {precision_score(y_test, y_pred):.3f}")
print(f"Recall    : {recall_score(y_test, y_pred):.3f}")
print(f"F1-Score  : {f1_score(y_test, y_pred):.3f}")
print(f"ROC AUC   : {roc_auc_score(y_test, y_proba):.3f}\n")

print("=== Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred), "\n")

print("=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=["Control (0)", "ADHD (1)"]))


Fitting 5 folds for each of 18 candidates, totalling 90 fits


=== Best Hyperparameters ===
{'C': 1, 'class_weight': 'balanced', 'gamma': 0.005, 'kernel': 'rbf'} 

=== Test Set Performance ===
Accuracy  : 0.720
Precision : 0.688
Recall    : 0.846
F1-Score  : 0.759
ROC AUC   : 0.788

=== Confusion Matrix ===
[[ 7  5]
 [ 2 11]] 

=== Classification Report ===
              precision    recall  f1-score   support

 Control (0)       0.78      0.58      0.67        12
    ADHD (1)       0.69      0.85      0.76        13

    accuracy                           0.72        25
   macro avg       0.73      0.71      0.71        25
weighted avg       0.73      0.72      0.71        25



In [25]:
# 1. Imports
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report
)

# 2. Load merged features & build target
df = pd.read_csv("merged_features.csv")
df["has_adhd"] = df["folder"].str.contains("ADHD", case=False).astype(int)

# 3. Split into train/test
X = df.drop(columns=["folder","filename","has_adhd"])
y = df["has_adhd"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 4. Standardize then LDA on TRAIN only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

lda = LinearDiscriminantAnalysis(n_components=1)
X_train_lda = lda.fit_transform(X_train_scaled, y_train)
X_test_lda  = lda.transform(X_test_scaled)

# 5. Define base learners
estimators = [
    ("dt", DecisionTreeClassifier(random_state=42)),
    ("rf", RandomForestClassifier(n_estimators=100, random_state=42)),
    ("svm", SVC(kernel="rbf", C=1, gamma=0.005, probability=True, class_weight="balanced", random_state=42)),
    ("lr", LogisticRegression(C=0.01, penalty="l2", max_iter=1000, random_state=42)),
    ("knn", KNeighborsClassifier(n_neighbors=3, weights="uniform"))
]

# 6. Build a soft‐voting ensemble
ensemble = VotingClassifier(
    estimators=estimators,
    voting="soft",        # use predict_proba averaging
    n_jobs=-1
)

# 7. Fit ensemble
ensemble.fit(X_train_lda, y_train)

# 8. Predict & evaluate on the held‐out test set
y_pred  = ensemble.predict(X_test_lda)
y_proba = ensemble.predict_proba(X_test_lda)[:, 1]

print("=== Ensemble Test Set Performance ===")
print(f"Accuracy  : {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision : {precision_score(y_test, y_pred):.3f}")
print(f"Recall    : {recall_score(y_test, y_pred):.3f}")
print(f"F1-Score  : {f1_score(y_test, y_pred):.3f}")
print(f"ROC AUC   : {roc_auc_score(y_test, y_proba):.3f}\n")

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred), "\n")

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Control (0)", "ADHD (1)"]))


=== Ensemble Test Set Performance ===
Accuracy  : 0.720
Precision : 0.688
Recall    : 0.846
F1-Score  : 0.759
ROC AUC   : 0.788

Confusion Matrix:
[[ 7  5]
 [ 2 11]] 

Classification Report:
              precision    recall  f1-score   support

 Control (0)       0.78      0.58      0.67        12
    ADHD (1)       0.69      0.85      0.76        13

    accuracy                           0.72        25
   macro avg       0.73      0.71      0.71        25
weighted avg       0.73      0.72      0.71        25

